In [6]:
import pandas as pd

file_path = 'dataset_ulasan_dana.csv'

try:
    # Memuat ulang data
    df = pd.read_csv(file_path)
    print(f"Berhasil memuat {len(df)} data.")

    # Menampilkan 5 data teratas untuk memastikan data masuk dengan benar
    display(df.head())
except FileNotFoundError:
    print("Error: File CSV tidak ditemukan! Pastikan file sudah diunggah ke environment.")

Berhasil memuat 11000 data.


,content,score,at,label
0,sangat berguna dan bermanfaat mendukung memuda...,5,2026-04-24 23:34:11,Positif
1,oke,5,2026-04-24 23:34:03,Positif
2,respon Diana si customer service ini gajelas m...,2,2026-04-24 23:24:29,Negatif
3,Mudah dan cepat buat kebutuhan sehari hari. Sa...,5,2026-04-24 23:18:25,Positif
4,"kembali error aplikasi, buka mistery box gagal...",3,2026-04-24 23:13:30,Netral


#Bagian 1: Text Preprocessing

In [7]:
!pip install Sastrawi
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [8]:
def clean_text(text):
    # 1. Case Folding (kecilkan semua huruf)
    text = text.lower()
    # 2. Hapus URL/Link
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # 3. Hapus Karakter Spesial, Angka, dan Emoji
    text = re.sub(r'[^a-z\s]', '', text)
    # 4. Hapus spasi berlebih
    text = text.strip()
    return text

# Terapkan ke dataset
df['content_cleaned'] = df['content'].apply(clean_text)

In [9]:
# kamus sederhana
norm_dict = {
    "yg": "yang", "gak": "tidak", "gk": "tidak", "tdk": "tidak",
    "bgt": "banget", "udah": "sudah", "sdh": "sudah", "jd": "jadi",
    "kalo": "kalau", "aplikasiny": "aplikasinya", "eror": "error"
}

def normalisasi(text):
    words = text.split()
    return " ".join([norm_dict.get(w, w) for w in words])

df['content_cleaned'] = df['content_cleaned'].apply(normalisasi)

In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 1. Mengubah label teks ke angka (Label Encoding)
# Negatif: 0, Netral: 1, Positif: 2
le = LabelEncoder()
y = le.fit_transform(df['label'])
# Ubah ke categorical (One-Hot Encoding) jika menggunakan categorical_crossentropy
from tensorflow.keras.utils import to_categorical
y = to_categorical(y)

# 2. Tokenisasi
max_words = 5000 # Ambil 5000 kata paling sering muncul
tokenizer = Tokenizer(num_words=max_words, split=' ')
tokenizer.fit_on_texts(df['content_cleaned'].values)
X = tokenizer.texts_to_sequences(df['content_cleaned'].values)

# 3. Padding (Menyamakan panjang kalimat)
X = pad_sequences(X, maxlen=100) # Batasi maksimal 100 kata per ulasan

# 4. Pembagian Data (Sesuai saran: 90% Training, 10% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Bagian 2: Modeling & Evaluation

# Model 1 : Deep learning menggunakan LSTM + Ekstraksi Fitur = Embedding Layer

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Dropout

model = Sequential()
model.add(Embedding(max_words, 128, input_length=X.shape[1]))
model.add(SpatialDropout1D(0.4))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(3, activation='softmax')) # 3 output untuk Positif, Netral, Negatif

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ ?                      │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [12]:
from tensorflow.keras.callbacks import EarlyStopping

epochs = 10
batch_size = 64

history = model.fit(X_train, y_train,
                    epochs=epochs,
                    batch_size=batch_size,
                    validation_data=(X_test, y_test),
                    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)])

Epoch 1/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 66s 366ms/step - accuracy: 0.8093 - loss: 0.5000 - val_accuracy: 0.8455 - val_loss: 0.4159
Epoch 2/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 72s 303ms/step - accuracy: 0.8547 - loss: 0.3984 - val_accuracy: 0.8645 - val_loss: 0.3876
Epoch 3/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 87s 336ms/step - accuracy: 0.8765 - loss: 0.3526 - val_accuracy: 0.8600 - val_loss: 0.3875
Epoch 4/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 76s 299ms/step - accuracy: 0.8898 - loss: 0.3195 - val_accuracy: 0.8645 - val_loss: 0.3955
Epoch 5/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 44s 286ms/step - accuracy: 0.9023 - loss: 0.2850 - val_accuracy: 0.8545 - val_loss: 0.4189
Epoch 6/10
155/155 ━━━━━━━━━━━━━━━━━━━━ 46s 296ms/step - accuracy: 0.9145 - loss: 0.2583 - val_accuracy: 0.8627 - val_loss: 0.4349


In [18]:
# Mengevaluasi model pada data testing
results = model.evaluate(X_test, y_test, verbose=0)

# Menampilkan hasil
print("--- HASIL EVALUASI MODEL (DEEP LEARNING) ---")
print(f"Loss pada Test Set    : {results[0]:.4f}")
print(f"Akurasi pada Test Set : {results[1] * 100:.2f}%")

# Mengevaluasi model pada data testing
results = model.evaluate(X_test, y_test, verbose=0)

# Menampilkan hasil
print("--- HASIL EVALUASI MODEL (DEEP LEARNING) ---")
print(f"Loss pada Test Set    : {results[0]:.4f}")
print(f"Akurasi pada Test Set : {results[1] * 100:.2f}%")

--- HASIL EVALUASI MODEL (DEEP LEARNING) ---
Loss pada Test Set    : 0.3875
Akurasi pada Test Set : 86.00%


In [13]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Inisialisasi StopWordRemoverFactory
factory = StopWordRemoverFactory()
stopword_list = factory.get_stop_words()
stopword_remover = factory.create_stop_word_remover()

def remove_stopwords(text):
    return stopword_remover.remove(text)

# Fungsi untuk mengetes kalimat baru
def predict_sentiment(text):
    cleaned = clean_text(text)
    norm = normalisasi(cleaned)
    sw = remove_stopwords(norm)
    seq = tokenizer.texts_to_sequences([sw])
    padded = pad_sequences(seq, maxlen=100)
    pred = model.predict(padded)
    labels = ['Negatif', 'Netral', 'Positif']
    return labels[np.argmax(pred)]

# Contoh Testing
test_text = "Aplikasi ini sangat membantu tapi loadingnya agak lambat."
print(f"Teks: {test_text}")
print(f"Prediksi Sentimen: {predict_sentiment(test_text)}")

Teks: Aplikasi ini sangat membantu tapi loadingnya agak lambat.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 396ms/step
Prediksi Sentimen: Positif


# Model 2 : SVM + TF-IDF (Split 80/20)

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. Persiapan Data (Menggunakan teks yang sudah dibersihkan)
X_text = df['content_cleaned']
y_label = df['label'] # Kita pakai label teks langsung (Positif, Netral, Negatif)

# 2. Ekstraksi Fitur: TF-IDF
# Menghitung bobot kata berdasarkan seberapa sering muncul di satu dokumen vs seluruh dokumen
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(X_text)

# 3. Pembagian Data: 80% Training, 20% Testing (Kriteria Skema 1)
X_train_sv, X_test_sv, y_train_sv, y_test_sv = train_test_split(
    X_tfidf, y_label, test_size=0.20, random_state=42
)

# 4. Pelatihan Model: SVM
# Kita gunakan kernel='linear' karena biasanya bekerja sangat baik untuk klasifikasi teks
svm_model = SVC(kernel='linear', C=1.0)
svm_model.fit(X_train_sv, y_train_sv)

# 5. Evaluasi
y_pred_sv = svm_model.fit(X_train_sv, y_train_sv).predict(X_test_sv)
acc_sv = accuracy_score(y_test_sv, y_pred_sv)

print("--- EVALUASI MODEL 2 (SVM + TF-IDF) ---")
print(f"Akurasi: {acc_sv * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test_sv, y_pred_sv))

--- EVALUASI MODEL 2 (SVM + TF-IDF) ---
Akurasi: 85.86%

Classification Report:
              precision    recall  f1-score   support

     Negatif       0.70      0.69      0.69       430
      Netral       0.00      0.00      0.00        85
     Positif       0.90      0.94      0.92      1685

    accuracy                           0.86      2200
   macro avg       0.53      0.55      0.54      2200
weighted avg       0.82      0.86      0.84      2200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
# Fungsi Inference SVM
def test_classic_models(kalimat):
    # Preprocessing
    teks_bersih = clean_text(kalimat)
    teks_norm = normalisasi(teks_bersih)
    teks_sw = remove_stopwords(teks_norm)
    # Inference
    vec_sv = tfidf.transform([teks_sw])
    pred_sv = svm_model.predict(vec_sv)[0]
    print(f"Kalimat: '{kalimat}'")
    print(f"Prediksi SVM : {pred_sv}")
    print("-" * 30)
# Uji Coba
print("--- TEST INFERENCE SVMt")
test_classic_models("Pelayanannya sangat mengecewakan dan lambat.")
test_classic_models("Terima kasih, transaksinya cepat sekali!")

--- TEST INFERENCE SVMt
Kalimat: 'Pelayanannya sangat mengecewakan dan lambat.'
Prediksi SVM : Negatif
------------------------------
Kalimat: 'Terima kasih, transaksinya cepat sekali!'
Prediksi SVM : Positif
------------------------------


# Model 3 Random Forest + CountVectorizer (Split 70/30)

In [16]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# 1. Persiapan Data
X_text_rf = df['content_cleaned']
y_label_rf = df['label']

# 2. Ekstraksi Fitur: CountVectorizer (Bag of Words)
# Berbeda dengan TF-IDF, ini hanya menghitung frekuensi kemunculan kata
cv = CountVectorizer(max_features=5000)
X_cv = cv.fit_transform(X_text_rf)

# 3. Pembagian Data: 70% Training, 30% Testing (Kriteria Skema 2)
X_train_rf, X_test_rf, y_train_rf, y_test_rf = train_test_split(
    X_cv, y_label_rf, test_size=0.30, random_state=42
)

# 4. Pelatihan Model: Random Forest
# Menggunakan 100 pohon keputusan (n_estimators)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_rf, y_train_rf)

# 5. Evaluasi
y_pred_rf = rf_model.predict(X_test_rf)
acc_rf = accuracy_score(y_test_rf, y_pred_rf)

print("--- EVALUASI Model 3 (Random Forest + CountVectorizer) ---")
print(f"Akurasi: {acc_rf * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test_rf, y_pred_rf))

--- EVALUASI Model 3 (Random Forest + CountVectorizer) ---
Akurasi: 85.45%

Classification Report:
              precision    recall  f1-score   support

     Negatif       0.69      0.69      0.69       675
      Netral       0.25      0.02      0.03       126
     Positif       0.90      0.94      0.92      2499

    accuracy                           0.85      3300
   macro avg       0.61      0.55      0.55      3300
weighted avg       0.83      0.85      0.84      3300



In [17]:
# Fungsi Inference Random Forest
def test_classic_models(kalimat):
    # Preprocessing
    teks_bersih = clean_text(kalimat)
    teks_norm = normalisasi(teks_bersih)
    teks_sw = remove_stopwords(teks_norm)
    # Inference
    vec_rf = cv.transform([teks_sw])
    pred_rf = rf_model.predict(vec_rf)[0]
    print(f"Kalimat: '{kalimat}'")
    print(f"Prediksi Random Forest : {pred_rf}")
    print("-" * 30)
# Uji Coba
print("--- TEST INFERENCE Random Forest")
test_classic_models("Pelayanannya sangat mengecewakan dan lambat.")
test_classic_models("Terima kasih, transaksinya cepat sekali!")

--- TEST INFERENCE Random Forest
Kalimat: 'Pelayanannya sangat mengecewakan dan lambat.'
Prediksi Random Forest : Positif
------------------------------
Kalimat: 'Terima kasih, transaksinya cepat sekali!'
Prediksi Random Forest : Positif
------------------------------
